# **Diagnostic Performance Analysis of an Assessment Tool for Predicting Cerebral Palsy (CP) - R**<br> 
The study aims to determine the accuracy of an assessment tool in predicting the development of cerebral palsy by one year of age in premature born children 

**Population** : Premature born children <br> 
**Index test** : an assessment tool measured at 3 different time points <br> 
**Outcome** : Incidence of Cerebral Palsy by one year of age

<u>**Main research questions answered in this analysis**</u> : 
- How well does the tool predicts development of CP? --> AUC, Specificity and Sensitivity 
- Which assessment time point provides the highest predictive accuracy for CP? --> Comparison of AUC
- Is there a significant different between the tool scores among CP and Non CP patients --> t-test/ wilcoxon 
- Is the predictive accuracy of the tool influenced by known confounding factors? --> Regression 


<u>**Variable names used in this code**</u> <br>
***Dependent Variable*** : CP/tidak (nomimal) : Development of Cerebral Palsy by one year age <br>
***Independent Variables*** : 
- Tool 1 (numerical) : score of the assessment tool measured at timepoint 1 
- Tool 2 (numerical) : score of the assessment tool measured at timepoint 2 
- Tool 3 (numerical) : score of the assessment tool measured at timepoint 3
- BBL (numerical) : birthweight 
- UG (numerical) : gestation age 
- NEC, BPD, IVH, PVL (nominal/ordinal) : neonatal complications 
- Surfaktan, PDA, Ventilasi, Inotropik, ROP, BERA (nominal/ordinal) : other confounding factors


<u>**Statistical Analysis and Data Visualization used in this study**</u>:
1. Univariate Analysis 
    - Frequency tables, Histogram, and Boxplots
    - AUC curves (and Youden Index) 
    - Sensitivity, Specificity
    - Normality test (shapiro) for numerical data
2. Bivariate Analysis
    - Frequency tables, Histogram, and Boxplots
    - Fisher exact test for categorical variables
    - t-test for normally distributed numerical variables
    - wilcoxon test for non-normally distributed numerical variables
3. Mutlivariate Analysis 
    - firth regression  
    - comparison of AUC values between three different time point assessment model <br>
<br>
- *fisher exact test was used due to several cells in frequency tables having <5 data, firth regression was used due to small sample size to address separation issues*
- *adjustments have been made to protect confidentiality of the data*

***Reproducibility Record*** <br>

R version 4.6.1 (2026-06-24) <br>
Platform : aarch64-apple-darwin23 <br> 
OS : macOS <br>

pROC 1.19.1 <br>
rio 1.3.0 <br>
logistf 1.26.1 <br>
tidyverse 2.0.0 <br>

**Data Cleaning**

In [ ]:
library(rio)
library(pROC)
library(logistf)
library(tidyverse)
setwd("insert your directory path")

raw <- import("nameoffile.csv")

#Cleaning Table 
#Deleting columns that are error due to excel format
raw <- raw[,-c(3,18,20,22, 24, 26, 31, 34,36, 40:47)]

View(raw)

#setting up column name
names(raw)[1:16] <- as.character(raw[3, 1:16])
names(raw)[17:28] <- as.character(raw[4, 17:28])
names(raw)[29:30] <- as.character(raw[3, 29:30])
raw <-  raw[-(1:4),] #deleting first 4 row that due to error
rownames(raw) <- NULL #reset col names
raw <- raw[1:72,] #capture all necessary rows

#renaming column name to easier name for analysis
raw <- raw %>%
  rename(
    `Tool 1` = `Tool 1 (SH/Y)`,
    `Tool 2` = `Tool 2 (SH/Y)`,
    `Tool 3` = `Tool 3 (SH/Y)`,
    `Length Stay` = `Lenght of stay`
  )

#converting data types for variables included in analysis
raw$`CP/tidak` <- factor(raw$`CP/tidak`,levels = c("Tidak CP", "CP"))  
raw$BBL <- as.numeric(raw$BBL)
raw$UG <- as.numeric(raw$UG)

#this data is formated as intraobserver value betwwen two researcher (ex 32/36), and we want to select the second observer's value
raw$`Tool 1` <- as.numeric(sub(".*/", "", raw$`Tool 1`))
raw$`Tool 2` <- as.numeric(sub(".*/", "", raw$`Tool 2`))
raw$`Tool 3` <- as.numeric(sub(".*/", "", raw$`Tool 3`))

raw$NEC <- factor(raw$NEC, levels = c("Tidak NEC", "NEC (+)"))
raw$Sepsis <- factor(raw$Sepsis, levels = c("Tidak Sepsis", "Sepsis (+)"))
raw$BPD <- factor(raw$BPD, levels = c("Tidak BPD", "BPD (+)"))

raw$IVH[raw$IVH %in% c("IVH grade 1", "IVH grade 2", "IVH grade 3")] <- "IVH"
raw$IVH <- factor(raw$IVH, levels = c("Tidak IVH", "IVH" ))

raw$PVL[raw$PVL %in% c("PVL grade 1", "PVL grade 2")] <- "PVL"
raw$PVL <- factor(raw$PVL, levels = c("Tidak PVL", "PVL"))

raw$Surfaktan <- factor(raw$Surfaktan, levels = c("tidak","ya"))
raw$PDA <- factor(raw$PDA, levels = c("tidak","ya"))
raw$Ventilasi <- factor(raw$Ventilasi, levels = c("Tidak","Ya"))
raw$Inotropik <- factor(raw$Inotropik, levels = c("tidak","ya"))

raw$ROP <- factor(raw$ROP, levels = c("Tidak","Ya"))
raw$BERA <- factor(raw$BERA, levels = c("BERA normal","BERA tidak normal"))
raw$`Length Stay` <- as.numeric(raw$`Length Stay`)

#Saving the cleaned original data into a csv and RDS 
write.csv(raw, "CSV_cleaned_data.csv", row.names = FALSE)
saveRDS(raw, "RDS_cleaned_data.rds", row.names = FALSE)

#Creating a df of only relevant variables
final <- raw %>% 
  select("No", "CP/tidak", "BBL", "UG", "Length Stay", "Tool 1", "Tool 2", "Tool 3", "NEC", "Sepsis", "BPD", "IVH", "PVL", "Surfaktan", "PDA", "Ventilasi", "Keterangan Ventilasi", "Inotropik", "ROP", "BERA")

#saving it into csv and RDS
write.csv(final, "CSV_ready_to_analyze.csv", row.names = FALSE)
saveRDS(final, "RDS_ready_to_analyze.rds")

**Univariate Analysis**

*Frequency Table*


In [ ]:
for (var in names(final)) {
  print(var)
  print(summary(final[,var]))
  cat("\n")
}

*Bar plot and Histogram*

In [ ]:
for (var in names(final)) {
  x <-  final[,var]
  if (is.numeric(x)) {
    hist(x, 
         main = var, 
         col = "skyblue")
  } else {
    barplot(table(x, useNA = "ifany"), 
            main = var, 
            col = "orange")
  }
}

#Creating a sidebyside boxplot for Tool value
par(mfrow = c(1,3))
boxplot(final$`Tool 1`, main = "Tool 1", ylim = c(35,80), col= "blue")
boxplot(final$`Tool 2`, main = "Tool 2", ylim = c(35,80), col = "orange")
boxplot(final$`Tool 3`, main = "Tool 3", ylim = c(35,80), col = "yellow")
par(mfrow = c(1,1))

*ROC curve and Diagnostic Indices*

In [ ]:
# Deleting patient with CP missing data
final <- final[!is.na(final$`CP/tidak`), ]

# Creating ROC curve, Calculating AUC, and other diagnostic indices
# For Tool 1
par(pty = "s")
#par(pty = "m")
roc_tool1 <- roc(final$`CP/tidak`, final$`Tool 1`, legacy.axes = TRUE)
auc_tool1 <-  auc(roc_tool1)
ci.auc(roc_tool1)

plot(roc_tool1,
     legacy.axes = TRUE,
     lwd = 2,
     main = "Tool 1",
     )

legend("bottomright",
       legend = paste("AUC =", round(auc_tool1, 3)),
       bty = "n"
       )

#Creating a table of specificity, sensitivity, and false positive percentage
roc.df1 <- data.frame(
    sen = roc_tool1$sensitivities * 100,
    spe = roc_tool1$specificities * 100,
    tpp = roc_tool1$sensitivities,
    fpp = (1 - roc_tool1$specificities),
    thresholds = roc_tool1$thresholds
    )

print(roc.df1)
coords(roc_tool1, "best", best.method = "youden")

# For Tool 2
roc_tool2 <- roc(final$`CP/tidak`, final$`Tool 2`, legacy.axes = TRUE)
auc_tool2 <- auc(roc_tool2)
ci.auc(roc_tool2)


plot(roc_tool2,
     legacy.axes = TRUE,
     lwd = 2,
     main = "Tool 2"
     )

legend("bottomright", 
      legend = paste("AUC =", round(auc_tool2, 3)), 
      bty = "n"
      )

roc.df2 <- data.frame(
    sen = roc_tool2$sensitivities * 100,
    spe = roc_tool2$specificities * 100,
    tpp = roc_tool2$sensitivities , 
    fpp = (1 - roc_tool2$specificities), 
    thresholds = roc_tool2$thresholds
    )

print(roc.df2)
coords(roc_tool2, "best", best.method = "youden")

# For Tool 3
roc_tool3 <- roc(final$`CP/tidak`, final$`Tool 3`, legacy.axes = TRUE)
auc_tool3 <- auc(roc_tool3)
                 
plot(roc_tool3,
     legacy.axes = TRUE,
     lwd = 2,
     main = "Tool 3"
     )

legend("bottomright", 
       legend = paste("AUC =", round(auc_tool3, 3)), 
       bty = "n"
      )

roc.df3 <- data.frame(
    sen = roc_tool3$sensitivities * 100, 
    spe = roc_tool3$specificities * 100,
    tpp = roc_tool3$sensitivities,
    fpp = (1 - roc_tool3$specificities),
    thresholds = roc_tool3$thresholds
    )

print(roc.df3)
coords(roc_tool3, "best", best.method = "youden")

par(pty = "m")

*Normality test*

In [ ]:
#Checking normality for numerical data
shapiro.test(final$UG) #result : not normally distributed
shapiro.test(final$BBL) #result : normally distributed
shapiro.test(final$`Tool 1`) #result : not normally distributed
shapiro.test(final$`Tool 2`) #result : not normally distributed
shapiro.test(final$`Tool 3`) #result : not normally distributed

**Bivariate Analysis**

*Frequency Tables*

In [ ]:
#Frequency table and fisher exact test for categorical variable
factor_vars <- names(final)[sapply(final, is.factor) & names(final) != "CP/tidak"]

*Boxplot*

In [ ]:
#creating boxplot for Tool variables
par(mfrow = c(1,3))
boxplot(`Tool 1` ~ `CP/tidak`, data = final, col = "blue", ylim = c(20,80))
boxplot(`Tool 2` ~ `CP/tidak`, data = final, col = "green", ylim = c(20,80))
boxplot(`Tool 3` ~ `CP/tidak`, data = final, col = "red", ylim = c(20,80))
par(mfrow = c(1,1))

*Fisher Exact Test*

In [ ]:
#Fisher exact test was used due to value <5 in count tables
for (v in factor_vars){
  print(v)
  tab <- (table(final[,v], final$`CP/tidak`))
  print(tab)
  cat("\n")
  print(fisher.test(tab))
} 

*Wilcoxon Test & t-test*

In [ ]:
#for not normally distributed data, we use wilcoxon test
#for normally distributed data, we use t-test 
tapply(final$UG, final$`CP/tidak`, summary)
wilcox.test(final$UG ~ final$`CP/tidak`)

tapply(final$BBL, final$`CP/tidak`, summary)
t.test(final$BBL ~ final$`CP/tidak`)

tapply(final$`Tool 1`, final$`CP/tidak`, summary)
wilcox.test(final$`Tool 1` ~ final$`CP/tidak`)

tapply(final$`Tool 2`, final$`CP/tidak`, summary)
wilcox.test(final$`Tool 2` ~ final$`CP/tidak`)

tapply(final$`Tool 3`, final$`CP/tidak`, summary)
wilcox.test(final$`Tool 3` ~ final$`CP/tidak`)

**Multivariate Analysis**

*Firth Regression* <br>

In [ ]:
##Multivariate analysis with confounding factors
fit_Tool1 <- logistf(`CP/tidak` ~ `Tool 1` + UG + BBL + NEC + Sepsis + BPD + IVH + PVL , data = final)
fit_Tool2 <- logistf(`CP/tidak` ~ `Tool 2` + UG + BBL + NEC + Sepsis + BPD + IVH + PVL , data = final)
fit_Tool3 <- logistf(`CP/tidak` ~ `Tool 3` + UG + BBL + NEC + Sepsis + BPD + IVH + PVL , data = final)
fit_combined <- logistf(`CP/tidak` ~ `Tool 1` + `Tool 2` + `Tool 3` + UG + BBL + NEC + Sepsis + BPD + IVH + PVL , data = final)

summary(fit_Tool1)
summary(fit_Tool2)
summary(fit_Tool3)
summary(fit_combined)

#generating table displaying multivariate analysis results 
make_result_firth <- function(model) {
  
  data.frame(
    Coef = round(model$coefficients, 3),
    Lower_CI_Coef = round(model$ci.lower, 3),
    Upper_CI_Coef = round(model$ci.upper, 3),
    OR = round(exp(model$coefficients), 3),
    Lower_CI_OR = round(exp(model$ci.lower), 3),
    Upper_CI_OR = round(exp(model$ci.upper), 3),
    p_value = round(model$prob, 3)
  )
}

result_Tool1 <- make_result_firth(fit_Tool1)
result_Tool2 <- make_result_firth(fit_Tool2)
result_Tool3 <- make_result_firth(fit_Tool3)
result_combined <- make_result_firth(fit_combined)

result_Tool1
result_Tool2 
result_Tool3 
result_combined

*AUC model comparison between different measurement timpoints (1 vs 2 vs 3)*

In [ ]:
#Comparing AUC values for Model with individual Tool scores and Combined Tool scores
#creating predicted probabilities
p1 <- predict(fit_Tool1, type = "response")
p2 <- predict(fit_Tool2, type = "response")
p3 <- predict(fit_Tool3, type = "response")
p_combined <- predict(fit_combined, type = "response")

#model.response extract outcome that was actually used in a fitted model
#regression can sometimes drop rows of data.  Roc requires same predictor and response
roc1 <- roc(model.response(model.frame(fit_Tool1)), p1)    
roc2 <- roc(model.response(model.frame(fit_Tool2)), p2)    
roc3 <- roc(model.response(model.frame(fit_Tool3)), p3)    
roc_combined <- roc(model.response(model.frame(fit_combined)), p_combined)    

#Displaying each model's AUC values
data.frame(
  Model = c("Tool 1", "Tool 2", "Tool 3", "Combined"),
  AUC = c(auc(roc1), auc(roc2), auc(roc3), auc(roc_combined))
)

#Displaying AUC curves 
par(mfrow =c(2,2))
plot(roc1, col = "blue", legacy.axes = TRUE, lwd = 2, main = "Tool 1 model")
legend("bottomright", 
       legend = paste("AUC =", round(auc(roc1), 3)), 
       bty = "n"
)

plot(roc2, col = "green", legacy.axes = TRUE, lwd = 2, main = "Tool 2 model")
legend("bottomright", 
       legend = paste("AUC =", round(auc(roc2), 3)), 
       bty = "n"
)

plot(roc3, col = "red", legacy.axes = TRUE, lwd = 2, main = "Tool 3 model")
legend("bottomright", 
       legend = paste("AUC =", round(auc(roc3), 3)), 
       bty = "n"
)

plot(roc_combined, col = "black", legacy.axes = TRUE, lwd = 2, main = "Combined (Tool 1, 2 and 3) model")
legend("bottomright", 
       legend = paste("AUC =", round(auc(roc_combined), 3)), 
       bty = "n"
)
par(mfrow =c(1,1))

**saving all environments and values for reproducibility**

In [ ]:
save.image("Environment_diagnostic")